In [4]:
import joblib
import numpy as np
import pandas as pd
import re
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# =========================================================
# 1. LOAD DATASET & TRAIN MODEL
# =========================================================
print("Loading Training.csv...")
df = pd.read_csv("Training.csv")

# Clean artifact column if present
if "Unnamed: 133" in df.columns:
    df.drop("Unnamed: 133", axis=1, inplace=True)

# Separate features (symptoms) and target (prognosis)
X = df.drop("prognosis", axis=1)
y = df["prognosis"]

print(df["prognosis"].value_counts().sum(), "total samples in dataset.")
# Encode target disease names
le = LabelEncoder()
y_encoded = le.fit_transform(y)


# Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# Initialize and Train Random Forest
print("Training Random Forest Classifier...")
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
)
rf.fit(X_train, y_train)

# Evaluate Accuracy
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {acc * 100:.2f}%")

# Save Models for Spring Boot / Deployment
# joblib.dump(rf, "disease_model.pkl")
# joblib.dump(le, "label_encoder.pkl")
print("Model files saved successfully as 'disease_model.pkl' and 'label_encoder.pkl'.\n")

# =========================================================
# 2. INITIALIZE NLP EMBEDDING ENGINE
# =========================================================
print("Loading NLP SentenceTransformer model (all-MiniLM-L6-v2)...")
nlp_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract feature names (132 symptoms)
feature_columns = list(X.columns)

# Convert column format from snake_case to natural readable text ('yellow_crust_ooze' -> 'yellow crust ooze')
formatted_columns = [col.replace("_", " ") for col in feature_columns]

# Pre-compute vector embeddings for all 132 symptom columns
column_embeddings = nlp_model.encode(formatted_columns, convert_to_tensor=True)


# =========================================================
# 3. RAW SENTENCE TO COLUMN MAPPER & PREDICTOR
# =========================================================
def predict_from_raw_text(
    raw_user_text: str, similarity_threshold: float = 0.45
):
    """Parses a natural sentence, maps phrases to exact dataset columns using cosine

    similarity, and predicts the disease.
    """
    if not raw_user_text or not raw_user_text.strip():
        return {"error": "Empty user input"}

    # 1. Regex clause extraction (split on punctuation & common linking words)
    pattern = r",|\.|\band\b|\bwith\b|\bsuffering from\b|\bi have\b"
    raw_phrases = re.split(pattern, raw_user_text.lower())
    phrases = [p.strip() for p in raw_phrases if p.strip()]

    if not phrases:
        return {"error": "No valid phrases detected"}

    # 2. Batch encoding for higher throughput
    phrase_embeddings = nlp_model.encode(
        phrases, convert_to_tensor=True, show_progress_bar=False
    )

    # 3. Pairwise Cosine Similarity (All phrases vs All columns)
    # Shape: (len(phrases), len(column_embeddings))
    cosine_scores = util.cos_sim(phrase_embeddings, column_embeddings)

    detected_symptoms = set()
    for row_scores in cosine_scores:
        # Get matching column indices exceeding threshold
        matched_indices = (
            torch.where(row_scores >= similarity_threshold)[0].cpu().numpy()
        )
        for idx in matched_indices:
            detected_symptoms.add(feature_columns[idx])

    detected_symptoms_list = list(detected_symptoms)

    # 4. Construct input vector efficiently
    input_data = {col: 0 for col in X.columns}
    for symptom in detected_symptoms_list:
        if symptom in input_data:
            input_data[symptom] = 1

    input_vector = pd.DataFrame([input_data])

    # 5. Model Prediction
    prediction = rf.predict(input_vector)[0]
    probabilities = rf.predict_proba(input_vector)[0]
    confidence = float(np.max(probabilities))

    # Handle label encoder mapping safely
    if hasattr(le, "inverse_transform"):
        predicted_disease = le.inverse_transform([prediction])[0]
    else:
        predicted_disease = str(prediction)

    return {
        "user_input": raw_user_text,
        "matched_columns": detected_symptoms_list,
        "predicted_disease": predicted_disease,
        "confidence_score": f"{round(confidence * 100, 2)}%",
    }

# =========================================================
# 4. TEST EXAMPLE
# =========================================================
if __name__ == "__main__":
    sample_sentence = "i have a high fever."

    print("--- Test Run ---")
    result = predict_from_raw_text(sample_sentence)

    print(f"User Input       : {result['user_input']}")
    print(f"Matched Columns  : {result['matched_columns']}")
    print(f"Predicted Disease: {result['predicted_disease']}")
    print(f"Confidence       : {result['confidence_score']}")

Loading Training.csv...
4920 total samples in dataset.
Training Random Forest Classifier...
Model Accuracy on Test Set: 100.00%
Model files saved successfully as 'disease_model.pkl' and 'label_encoder.pkl'.

Loading NLP SentenceTransformer model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- Test Run ---
User Input       : i have a high fever.
Matched Columns  : ['high_fever', 'shivering', 'mild_fever']
Predicted Disease: Allergy
Confidence       : 38.0%
